<h1> Primary data collection for oil production in Texas <h1>

All necessary data for texas oil procuction is obtained from https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/. We will be using the Production Data Query Dump dataset. This program extracts the necessary information from OG_LEASE_CYCLE_DATA_TABLE.dsv, which has oil and gas production data for every leases from 1993 to 2016 and from OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv, which has information about how the oil is dispensed out of the production plant. This program ignores gas wells and only focusses on the oil wells. The gas data available here is from the Casinghead gas, which is the natural gas that is found dissolved in crude oil and is produced alongside it (often flared).

Texas data files do not track information at the level of oil wells (at least the ones available for public). So all the production information is at the level of leases which may contain more than one well.

In [5]:
# ── CELL 1: Imports + Config ─────────────────────────────────────────────────
# Run this first after any kernel crash. Everything needed is in this one cell.

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

OUTER_ZIP    = "../../../../data/raw/texas/texas_pdq.zip"     # ← path to your outer zip
INNER_ZIP    = "PDQ_DSV.zip"                               # name of zip inside outer zip
OUT_DIR      = "../../../../data/raw/texas/cleaned_data"  # where to save outputs
FORMAT       = "parquet"                                   # "parquet" or "csv"
CHUNKSIZE    = 150_000                                     # reduced for 16GB RAM
OIL_GAS_FILTER = "O"                                       # "O" = oil leases only
                                                           # "G" = gas leases only
                                                           # None = load everything
DO_MERGE     = True                                        # True = produce joined table

In [6]:
# ── CELL 2: Constants ─────────────────────────────────────────────────────────

DELIMITER  = "}"
ENCODING   = "latin-1"
CYCLE_FILE = "OG_LEASE_CYCLE_DATA_TABLE.dsv"
DISP_FILE  = "OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv"

CYCLE_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",           
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "LEASE_OIL_PROD_VOL",
    "LEASE_CSGD_PROD_VOL",
    "LEASE_CSGD_TOT_DISP",
    "OPERATOR_NO",
    "OPERATOR_NAME",
]

OIL_DISP_LABELS = {
    "LEASE_OIL_DISPCD00_VOL": "oil_pipeline_bbl",
    "LEASE_OIL_DISPCD01_VOL": "oil_truck_bbl",
    "LEASE_OIL_DISPCD02_VOL": "oil_tankcar_bbl",
    "LEASE_OIL_DISPCD03_VOL": "oil_tank_cleaning_bbl",
    "LEASE_OIL_DISPCD04_VOL": "oil_circulating_bbl",
    "LEASE_OIL_DISPCD05_VOL": "oil_lost_stolen_bbl",
    "LEASE_OIL_DISPCD06_VOL": "oil_bsw_repressure_bbl",
    "LEASE_OIL_DISPCD07_VOL": "oil_legacy_bbl",
    "LEASE_OIL_DISPCD08_VOL": "oil_skimmed_bbl",
    "LEASE_OIL_DISPCD09_VOL": "oil_scrubber_bbl",
    "LEASE_OIL_DISPCD99_VOL": "oil_no_disp_code_bbl",
}


CSGD_DISP_LABELS = {
    "LEASE_CSGD_DISPCDE01_VOL": "csgd_field_ops_fuel_mcf",
    "LEASE_CSGD_DISPCDE02_VOL": "csgd_transmission_mcf",
    "LEASE_CSGD_DISPCDE03_VOL": "csgd_processing_plant_mcf",
    "LEASE_CSGD_DISPCDE04_VOL": "csgd_vented_flared_mcf",
    "LEASE_CSGD_DISPCDE05_VOL": "csgd_gas_lift_mcf",
    "LEASE_CSGD_DISPCDE06_VOL": "csgd_repressure_mcf",
    "LEASE_CSGD_DISPCDE07_VOL": "csgd_carbon_black_mcf",
    "LEASE_CSGD_DISPCDE08_VOL": "csgd_underground_storage_mcf",
    "LEASE_CSGD_DISPCDE99_VOL": "csgd_no_disp_code_mcf",
}
ALL_DISP_LABELS = {
    **OIL_DISP_LABELS, **CSGD_DISP_LABELS,
}

DISP_KEYS = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",           
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "OPERATOR_NO",
    "OPERATOR_NAME",
]
DISP_KEEP = DISP_KEYS + list(ALL_DISP_LABELS.keys())

JOIN_KEYS = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",           
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "OPERATOR_NO",
    "OPERATOR_NAME",
]

log.info("✓ Cell 2 done — constants loaded.")


14:22:39 [INFO] ✓ Cell 2 done — constants loaded.


In [7]:
# ── CELL 3: Cleaning Functions ────────────────────────────────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def _cast_cycle(df):
    num_cols = [c for c in df.columns if any(c.startswith(p) for p in
                ("LEASE_OIL_", "LEASE_GAS_", "LEASE_COND_", "LEASE_CSGD_"))]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    ym = df["CYCLE_YEAR_MONTH"].astype(str).str.zfill(6)
    df["PROD_DATE"] = pd.to_datetime(
        ym.str[:4] + "-" + ym.str[4:] + "-01", errors="coerce")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "FIELD_TYPE", "PROD_REPORT_FILED_FLAG"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _cast_disp(df):
    for col in [c for c in df.columns if "_DISPCD" in c or "_DISPCDE" in c]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _add_derived_disp_cols(df):
    present = set(df.columns)
    sold = ["LEASE_OIL_DISPCD00_VOL", "LEASE_OIL_DISPCD01_VOL", "LEASE_OIL_DISPCD02_VOL"]
    if all(c in present for c in sold):
        df["oil_sold_total_bbl"] = df[sold].sum(axis=1, min_count=1)
    g_flare, cg_flare = "LEASE_GAS_DISPCD04_VOL", "LEASE_CSGD_DISPCDE04_VOL"
    if g_flare in present and cg_flare in present:
        df["total_vented_flared_mcf"] = df[[g_flare, cg_flare]].sum(axis=1, min_count=1)
    elif g_flare in present:
        df["total_vented_flared_mcf"] = df[g_flare]
    elif cg_flare in present:
        df["total_vented_flared_mcf"] = df[cg_flare]
    g_proc, cg_proc = "LEASE_GAS_DISPCD03_VOL", "LEASE_CSGD_DISPCDE03_VOL"
    if g_proc in present and cg_proc in present:
        df["total_gas_to_processing_mcf"] = df[[g_proc, cg_proc]].sum(axis=1, min_count=1)
    return df

def clean_cycle_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in CYCLE_KEEP if c in chunk.columns]]
    chunk = _cast_cycle(chunk)
    vol_cols = [c for c in chunk.columns if c.endswith("_PROD_VOL")]
    chunk[vol_cols] = chunk[vol_cols].replace(0, pd.NA)
    return chunk

def clean_disp_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in DISP_KEEP if c in chunk.columns]]
    chunk = _cast_disp(chunk)
    chunk = _add_derived_disp_cols(chunk)
    chunk = chunk.rename(columns={
        k: v for k, v in ALL_DISP_LABELS.items() if k in chunk.columns})
    return chunk

log.info("✓ Cell 3 done — cleaning functions defined.")


14:22:39 [INFO] ✓ Cell 3 done — cleaning functions defined.


In [8]:
# ── CELL 4: Reader Functions ──────────────────────────────────────────────────

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found in outer zip.\nAvailable: {outer_contents}")
        log.info("Opening inner zip: %s", match)
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Cell 4 done — reader functions defined.")


14:22:40 [INFO] ✓ Cell 4 done — reader functions defined.


In [9]:
# ── CELL 5: Load OG_LEASE_CYCLE ───────────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_cycle = read_chunked(inner_zf, CYCLE_FILE, clean_cycle_chunk)
inner_zf.close()
gc.collect()

print("\nShape      :", df_cycle.shape)
print("Date range :", df_cycle["PROD_DATE"].min(), "→", df_cycle["PROD_DATE"].max())
print("Memory     :", f"{df_cycle.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns    :", df_cycle.columns.tolist())
df_cycle.head(3)


14:22:42 [INFO] 
── Loading OG_LEASE_CYCLE ──
14:22:42 [INFO] Files in outer zip: ['PDQ_DSV.zip']
14:22:42 [INFO] Opening inner zip: PDQ_DSV.zip
14:22:46 [INFO] Reading OG_LEASE_CYCLE_DATA_TABLE.dsv (chunk size = 150,000) ...
14:22:47 [INFO]   chunk   1 — kept 44,754 / 150,000 rows
14:22:47 [INFO]   chunk   2 — kept 99,294 / 300,000 rows
14:22:47 [INFO]   chunk   3 — kept 156,248 / 450,000 rows
14:22:48 [INFO]   chunk   4 — kept 225,294 / 600,000 rows
14:22:48 [INFO]   chunk   5 — kept 280,138 / 750,000 rows
14:22:48 [INFO]   chunk   6 — kept 368,123 / 900,000 rows
14:22:49 [INFO]   chunk   7 — kept 423,924 / 1,050,000 rows
14:22:49 [INFO]   chunk   8 — kept 508,491 / 1,200,000 rows
14:22:50 [INFO]   chunk   9 — kept 561,626 / 1,350,000 rows
14:22:50 [INFO]   chunk  10 — kept 596,273 / 1,500,000 rows
14:22:50 [INFO]   chunk  11 — kept 636,592 / 1,650,000 rows
14:22:51 [INFO]   chunk  12 — kept 705,075 / 1,800,000 rows
14:22:51 [INFO]   chunk  13 — kept 759,134 / 1,950,000 rows
14:22:51

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,CYCLE_YEAR_MONTH,FIELD_NO,LEASE_OIL_PROD_VOL,LEASE_CSGD_PROD_VOL,LEASE_CSGD_TOT_DISP,OPERATOR_NO,OPERATOR_NAME,PROD_DATE
0,O,02,05905,202601,57911234,<NA>,<NA>,0,222226,"DOLPHIN PETROLEUM, LP",2026-01-01
1,O,02,05905,202602,57911234,<NA>,<NA>,0,222226,"DOLPHIN PETROLEUM, LP",2026-02-01
2,O,02,05905,202603,57911234,<NA>,<NA>,0,222226,"DOLPHIN PETROLEUM, LP",2026-03-01


In [ ]:
#Make a date_time column
print(df_cycle.columns)
df_cycle['date'] = pd.to_datetime(df_cycle['CYCLE_YEAR_MONTH'],format = '%Y%m')
df_cycle.date.head(3)

Index(['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'CYCLE_YEAR_MONTH',
       'FIELD_NO', 'LEASE_OIL_PROD_VOL', 'LEASE_CSGD_PROD_VOL',
       'LEASE_CSGD_TOT_DISP', 'OPERATOR_NO', 'OPERATOR_NAME', 'PROD_DATE'],
      dtype='str')


0   2026-01-01
1   2026-02-01
2   2026-03-01
Name: date, dtype: datetime64[us]

In [18]:
df_cycle = df_cycle.drop(columns = ['CYCLE_YEAR_MONTH','PROD_DATE'])

In [19]:
# Do further cleaning to make sure that NA for oil production is exluded.
df_cycle_clean = df_cycle.dropna(subset = 'LEASE_OIL_PROD_VOL').reset_index(drop=True).copy()

In [20]:
df_cycle_clean.head(3)

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,FIELD_NO,LEASE_OIL_PROD_VOL,LEASE_CSGD_PROD_VOL,LEASE_CSGD_TOT_DISP,OPERATOR_NO,OPERATOR_NAME,date
0,O,03,24870,32688470,115,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-01-01
1,O,03,24870,32688470,349,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-02-01
2,O,03,24870,32688470,203,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-03-01


In [21]:
df_cycle_clean.columns = df_cycle_clean.columns.str.lower()

In [22]:
df_cycle_clean.head(3)

,oil_gas_code,district_no,lease_no,field_no,lease_oil_prod_vol,lease_csgd_prod_vol,lease_csgd_tot_disp,operator_no,operator_name,date
0,O,03,24870,32688470,115,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-01-01
1,O,03,24870,32688470,349,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-02-01
2,O,03,24870,32688470,203,<NA>,0,386310,HILCORP ENERGY COMPANY,2026-03-01


In [23]:
df_cycle_clean.operator_name.unique()

<ArrowStringArray>
[          'HILCORP ENERGY COMPANY', 'DORCHESTER OPERATING COMPANY LLC',
                 'IV STANDARD, LLC',      'VERMILLION PRODUCTION, INC.',
             'T-C OIL COMPANY, LLC',       'CENTRAL GULF PETROLEUM INC',
        'WHITE ROCK OIL & GAS, LLC',           'ALDINE OIL AND GAS, LP',
             'SHOCO PRODUCTION LLC',         'FOURCOOKS OIL & GAS, LLC',
 ...
       'EQUITABLE ENERGY, INC. (I)',                'B & F INVESTMENTS',
                   'RAGLE, WILLIAM',   'SOUTHERN UNION EXPLORATION CO.',
                   'HOLLIS OIL CO.',     'SIGMA EXPLORATION CORP., THE',
           'KING, AL PETROLEUM CO.',               'FARMER OIL COMPANY',
                 'CAMPBELL OIL CO.',                   'L-D PRODUCTION']
Length: 17130, dtype: str

In [24]:
# Saving this file
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

def save_df(df, name):
    path = out / f"{name}.{FORMAT}"
    if FORMAT == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    log.info("Saved %s  (%.1f MB)", path.name, path.stat().st_size / 1_048_576)

log.info("Saving df_cycle ...")
save_df(df_cycle_clean, "texas_total_prod")
del df_cycle
gc.collect()
log.info("✓ df_cycle_clean saved and freed from memory.")

14:33:11 [INFO] Saving df_cycle ...
14:33:14 [INFO] Saved texas_total_prod.parquet  (129.2 MB)
14:33:14 [INFO] ✓ df_cycle_clean saved and freed from memory.


In [25]:
del df_cycle_clean

In [26]:
test = pd.read_parquet("../../../../data/raw/texas/cleaned_data/texas_total_prod.parquet")

In [27]:
test.sample(20)
del test

In [28]:
log.info("\n── Loading OG_LEASE_CYCLE_DISP ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_disp = read_chunked(inner_zf, DISP_FILE, clean_disp_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_disp.shape)
print("Memory :", f"{df_disp.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_disp.columns.tolist())

dupes = df_disp.columns[df_disp.columns.duplicated()].tolist()
if dupes:
    log.error("Duplicate columns in df_disp: %s", dupes)
else:
    log.info("✓ No duplicate columns.")

df_disp.head(3)


14:34:04 [INFO] 
── Loading OG_LEASE_CYCLE_DISP ──
14:34:04 [INFO] Files in outer zip: ['PDQ_DSV.zip']
14:34:04 [INFO] Opening inner zip: PDQ_DSV.zip
14:34:10 [INFO] Reading OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv (chunk size = 150,000) ...
14:34:11 [INFO]   chunk   1 — kept 47,657 / 150,000 rows
14:34:12 [INFO]   chunk   2 — kept 108,852 / 300,000 rows
14:34:13 [INFO]   chunk   3 — kept 148,948 / 450,000 rows
14:34:13 [INFO]   chunk   4 — kept 195,204 / 600,000 rows
14:34:14 [INFO]   chunk   5 — kept 233,180 / 750,000 rows
14:34:15 [INFO]   chunk   6 — kept 283,864 / 900,000 rows
14:34:16 [INFO]   chunk   7 — kept 346,512 / 1,050,000 rows
14:34:16 [INFO]   chunk   8 — kept 392,278 / 1,200,000 rows
14:34:17 [INFO]   chunk   9 — kept 481,437 / 1,350,000 rows
14:34:18 [INFO]   chunk  10 — kept 525,218 / 1,500,000 rows
14:34:18 [INFO]   chunk  11 — kept 591,342 / 1,650,000 rows
14:34:19 [INFO]   chunk  12 — kept 666,076 / 1,800,000 rows
14:34:20 [INFO]   chunk  13 — kept 710,671 / 1,950,000 ro

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,CYCLE_YEAR_MONTH,FIELD_NO,OPERATOR_NO,OPERATOR_NAME,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,...,csgd_transmission_mcf,csgd_processing_plant_mcf,csgd_vented_flared_mcf,csgd_gas_lift_mcf,csgd_repressure_mcf,csgd_carbon_black_mcf,csgd_underground_storage_mcf,csgd_no_disp_code_mcf,oil_sold_total_bbl,total_vented_flared_mcf
0,O,08,09073,202603,89812001,101688,"FOURCOOKS OIL & GAS, LLC",0,165,0,...,0,0,0,0,0,0,0,0,165,0
1,O,10,27510,202601,19541001,623254,ONE NICKEL OPERATING LLC,0,146,0,...,0,0,0,0,0,0,0,0,146,0
2,O,10,27510,202602,19541001,623254,ONE NICKEL OPERATING LLC,0,154,0,...,0,0,0,0,0,0,0,0,154,0


In [30]:
#Make a date_time column
print(df_disp.columns)
df_disp['date'] = pd.to_datetime(df_disp['CYCLE_YEAR_MONTH'],format = '%Y%m')
df_disp.date.head(3)

Index(['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'CYCLE_YEAR_MONTH',
       'FIELD_NO', 'OPERATOR_NO', 'OPERATOR_NAME', 'oil_pipeline_bbl',
       'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl',
       'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl',
       'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl',
       'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf',
       'csgd_transmission_mcf', 'csgd_processing_plant_mcf',
       'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf',
       'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf',
       'csgd_no_disp_code_mcf', 'oil_sold_total_bbl',
       'total_vented_flared_mcf'],
      dtype='str')


0   2026-03-01
1   2026-01-01
2   2026-02-01
Name: date, dtype: datetime64[us]

In [33]:
df_disp = df_disp.drop(columns = ['CYCLE_YEAR_MONTH'])

In [34]:
# Cleaning further
df_disp_clean = df_disp.dropna(subset = 'oil_sold_total_bbl').reset_index(drop=True).copy()
df_disp_clean.head(3)

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,FIELD_NO,OPERATOR_NO,OPERATOR_NAME,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,...,csgd_processing_plant_mcf,csgd_vented_flared_mcf,csgd_gas_lift_mcf,csgd_repressure_mcf,csgd_carbon_black_mcf,csgd_underground_storage_mcf,csgd_no_disp_code_mcf,oil_sold_total_bbl,total_vented_flared_mcf,date
0,O,08,09073,89812001,101688,"FOURCOOKS OIL & GAS, LLC",0,165,0,0,...,0,0,0,0,0,0,0,165,0,2026-03-01
1,O,10,27510,19541001,623254,ONE NICKEL OPERATING LLC,0,146,0,0,...,0,0,0,0,0,0,0,146,0,2026-01-01
2,O,10,27510,19541001,623254,ONE NICKEL OPERATING LLC,0,154,0,0,...,0,0,0,0,0,0,0,154,0,2026-02-01


In [36]:
df_disp_clean.columns = df_disp_clean.columns.str.lower()

In [37]:
log.info("Saving df_disp ...")
save_df(df_disp_clean, "texas_prod_disp")
del df_disp
del df_disp_clean
gc.collect()
log.info("✓ df_disp_clean saved and freed from memory.")

14:41:44 [INFO] Saving df_disp ...
14:41:49 [INFO] Saved texas_prod_disp.parquet  (157.5 MB)
14:41:50 [INFO] ✓ df_disp_clean saved and freed from memory.
